### Análise Exploratória das Métricas Globais de Conectividade

Este notebook apresenta a análise exploratória das métricas globais de conectividade funcional cerebral extraídas a partir de sinais de EEG em estado de repouso.

O objetivo desta etapa é:
- caracterizar estatisticamente as métricas globais,
- investigar diferenças entre condições experimentais (EO vs EC),
- fornecer subsídios para as etapas posteriores de redução de dimensionalidade e aprendizado de máquina.


### Import e carregamento

In [1]:
import pandas as pd
import numpy as np
import plotly.io as pio

import plotly.express as px
import plotly.graph_objects as go

In [2]:
pio.templates.default = "plotly_dark"

In [3]:
# Carrega dataset final
df_dataset = pd.read_parquet("../../data/processed/dataset_features.parquet")

df_dataset.shape

(108, 360)

In [4]:
df_dataset

,degree_AF3,clustering_AF3,betweenness_AF3,hub_frequency_AF3,sd_degree_AF3,bin_clustering_AF3,degree_AF4,clustering_AF4,betweenness_AF4,hub_frequency_AF4,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
0,0.421064,0.389635,0.001299,0.003559,0.185907,0.658822,0.434218,0.400541,0.000000,0.002542,...,False,False,False,False,True,False,True,True,False,False
1,0.456114,0.424352,0.000000,0.001017,0.197066,0.707606,0.447632,0.415106,0.003133,0.001017,...,False,False,False,False,True,False,True,True,False,False
2,0.585267,0.572429,0.000000,0.008134,0.221862,0.709774,0.566095,0.558080,0.000000,0.004067,...,False,False,False,False,True,False,True,False,True,True
3,0.440010,0.433551,0.000565,0.004067,0.209431,0.669874,0.422954,0.421807,0.000000,0.003559,...,False,False,False,False,True,False,True,False,True,True
4,0.665150,0.640770,0.000000,0.000000,0.182402,0.793544,0.675006,0.647611,0.000000,0.000000,...,False,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,0.413301,0.418309,0.000000,0.005592,0.220927,0.656618,0.415076,0.417131,0.004040,0.000000,...,False,False,False,False,True,True,False,True,False,True
104,0.465274,0.486956,0.000000,0.000000,0.225301,0.734458,0.368751,0.420833,0.000000,0.001017,...,False,False,False,False,True,False,True,True,False,False
105,0.428436,0.446453,0.000565,0.001525,0.224430,0.666602,0.389714,0.413874,0.001130,0.008643,...,False,False,False,False,True,False,True,True,False,False
106,0.608886,0.605972,0.000000,0.000508,0.233453,0.760082,0.581683,0.589138,0.000000,0.000508,...,False,False,False,False,True,False,True,True,False,True


In [5]:
df_dataset['condition'].value_counts()

condition
EC    54
EO    54
Name: count, dtype: int64

### Seleção das métricas globais

“As métricas globais representam propriedades agregadas da rede funcional cerebral, independententes de regiões específicas, refletindo o nível geral de integração e variabilidade da conectividade.”

In [6]:
import pandas as pd

GLOBAL_FEATURES = [
    "edges_mean",
    "edges_std",
    "edges_cv",
]

id_cols = ["subject_id", "condition"]

if "df_dataset" not in globals():
    df_dataset = pd.read_parquet("../../data/processed/dataset_features.parquet")

df_global = df_dataset[id_cols + GLOBAL_FEATURES]

df_global.head()

,subject_id,condition,edges_mean,edges_std,edges_cv
0,sub-010069,EC,676.759024,183.904835,0.271743
1,sub-010069,EO,758.577529,223.862323,0.295108
2,sub-010070,EC,1038.072191,263.629913,0.253961
3,sub-010070,EO,929.931876,257.860633,0.277290
4,sub-010079,EC,1041.876462,228.158309,0.218988


In [7]:
df_global[GLOBAL_FEATURES].describe()

,edges_mean,edges_std,edges_cv
count,108.000000,108.000000,108.000000
mean,1010.638894,231.663170,0.232085
std,181.040071,36.346427,0.030794
min,563.887138,135.826090,0.169838
25%,890.809100,203.698532,0.207837
50%,1013.819268,231.003233,0.228724
75%,1136.331215,261.373493,0.251632
max,1430.381800,313.340638,0.309778


In [8]:
df_global.groupby("condition")[GLOBAL_FEATURES].describe()

edges_mean                                                   \
               count         mean         std         min         25%   
condition                                                               
EC              54.0  1037.990444  166.556749  660.156075  932.953991   
EO              54.0   983.287343  192.103683  563.887138  876.444967   

                                                 edges_std              ...  \
                   50%          75%          max     count        mean  ...   
condition                                                               ...   
EC         1047.060244  1163.147941  1379.859685      54.0  232.534949  ...   
EO          987.278088  1103.610702  1430.381800      54.0  230.791392  ...   

                                  edges_cv                                \
                  75%         max    count      mean       std       min   
condition                                                                  
EC         261.509876  313.340638     54.0  0.226178  0.030607  0.169838   
EO         260.455749  292.312279     54.0  0.237993  0.030109  0.176229   

                                                   
                25%       50%       75%       max  
condition                                          
EC         0.200763  0.218474  0.248122  0.309778  
EO         0.217390  0.238533  0.259773  0.303472  

[2 rows x 24 columns]

### Boxplots EO vs EC

Número médio de arestas

In [9]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_mean",
    color="condition",
    points="all",
    title="Número Médio de Arestas por Condição"
)

fig.show()

### Variabilidade da conectividade

In [10]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_cv",
    color="condition",
    points="all",
    title="Coeficiente de Variação da Conectividade ao Longo do Tempo"
)
fig.show()

### Correlação entre métricas globais

In [11]:
corr = df_global[GLOBAL_FEATURES].corr()
corr

,edges_mean,edges_std,edges_cv
edges_mean,1.000000,0.695386,-0.523477
edges_std,0.695386,1.000000,0.234561
edges_cv,-0.523477,0.234561,1.000000


In [12]:
fig = px.imshow(
    corr,
    text_auto=True,
    title="Matriz de Correlação entre Métricas Globais",
    color_continuous_scale="RdBu_r",
    aspect="auto"
)
fig.show()